# Loan Eligibility Prediction System - End-to-End ML Pipeline
**Course/Project Submission: B.Tech Machine Learning**

---

### System Overview
This project implements a complete, production-quality Machine Learning system to predict loan eligibility based on applicant profiles.
The workflow strictly follows standard ML lifecycle best practices:

1. **Data Ingestion**: Kaggle Loan Prediction Dataset (614 records, 13 features).
2. **Sequential Preprocessing**:
   - Missing value imputation (median for continuous numeric, mode for binary/categorical).
   - Label encoding for categorical variables (`LabelEncoder`).
   - Feature scaling for continuous numeric features (`StandardScaler`).
   - Stratified train-test split (80:20 ratio, `random_state=42`).
3. **Model Training & Comparison**:
   - Logistic Regression
   - Decision Tree Classifier
   - Random Forest Classifier
   - Support Vector Machine (SVM)
   - XGBoost Classifier (Gradient Boosting)
4. **Evaluation & Visualization**:
   - Accuracy, Precision, Recall, F1 Score, Confusion Matrix, ROC AUC.
   - Model accuracy comparison bar chart.
   - Best model (XGBoost) confusion matrix & ROC curve.
   - Combined ROC curves for all algorithms.
5. **Model Persistence & Inference**:
   - Save trained XGBoost model and preprocessors using `joblib`.
   - Sample inference pipeline for real-time prediction.


In [ ]:
# Step 0: Import Required Standard Libraries
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import xgboost as xgb
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_curve,
    auc,
    roc_auc_score
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
print("[INFO] All required libraries imported successfully.")

from sklearn.utils import resample


## Step 1: Data Ingestion & Exploration

In [ ]:
# Load dataset from local file or raw GitHub URL
file_path = 'loan_data.csv'
raw_url = 'https://raw.githubusercontent.com/shrikant-temburwar/Loan-Prediction-Dataset/master/train.csv'

if os.path.exists(file_path):
    print(f"[INFO] Loading dataset from local file: '{file_path}'")
    df = pd.read_csv(file_path)
else:
    print(f"[INFO] Fetching dataset from GitHub URL...")
    df = pd.read_csv(raw_url)
    df.to_csv(file_path, index=False)

print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


## Step 2: Data Preprocessing Pipeline (Sequential)

In [ ]:
# 0. Check & Ensure 50/50 Target Class Balance
print("--- Target Class Balance Inspection ---")
counts = df['Loan_Status'].value_counts()
pcts = df['Loan_Status'].value_counts(normalize=True) * 100
for cls, count in counts.items():
    print(f"  Class '{cls}': {count} samples ({pcts[cls]:.1f}%)")

if counts.min() < counts.max():
    print("[INFO] Target class is imbalanced. Resampling minority class to achieve 50/50 split...")
    dfs_balanced = []
    max_c = counts.max()
    for cls, group in df.groupby('Loan_Status'):
        if len(group) < max_c:
            dfs_balanced.append(resample(group, replace=True, n_samples=max_c, random_state=42))
        else:
            dfs_balanced.append(group)
    df = pd.concat(dfs_balanced).sample(frac=1, random_state=42).reset_index(drop=True)
    print("[SUCCESS] Dataset balanced to 50/50 split.")
else:
    print("[SUCCESS] Dataset already has an exact 50/50 balanced split.")

# Drop identifier column 'Loan_ID' if present
if 'Loan_ID' in df.columns:
    df = df.drop(columns=['Loan_ID'])
    print("[INFO] Dropped 'Loan_ID' identifier column.")

# 1. Missing Value Imputation
# Impute nulls in ApplicantIncome, CoapplicantIncome, LoanAmount with median
numeric_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount']
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Impute Credit_History and other categoricals with mode
categorical_impute_cols = ['Credit_History', 'Gender', 'Married', 'Dependents', 
                          'Education', 'Self_Employed', 'Property_Area', 'Loan_Amount_Term']
for col in categorical_impute_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print(f"Missing values remaining after imputation: {df.isnull().sum().sum()}")

# 2. Label Encoding
categorical_fields = ['Gender', 'Married', 'Education', 'Self_Employed', 
                      'Property_Area', 'Dependents', 'Loan_Status']
label_encoders = {}
for col in categorical_fields:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

# 3. Feature Scaling
num_scale_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term']
scaler = StandardScaler()
df[num_scale_cols] = scaler.fit_transform(df[num_scale_cols])

# 4. Stratified Train-Test Split (80:20)
X = df.drop(columns=['Loan_Status'])
y = df['Loan_Status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training Set Shape: {X_train.shape}, Test Set Shape: {X_test.shape}")
print(f"Test target balance: Rejected(0)={np.sum(y_test==0)}, Approved(1)={np.sum(y_test==1)}")


## Step 3: Model Training across 5 Algorithms

In [ ]:
# Initialize and train all 5 Machine Learning models
models = {
    'Logistic Regression': LogisticRegression(C=1.0, solver='lbfgs', random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(criterion='gini', max_depth=3, min_samples_split=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=4, min_samples_split=2, random_state=42),
    'SVM': SVC(C=1.0, kernel='rbf', probability=True, random_state=42),
    'XGBoost': xgb.XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, subsample=0.8, colsample_bytree=0.8, gamma=0.1, random_state=42, eval_metric='logloss')
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"  ✓ {name} trained successfully.")


## Step 4: Model Evaluation & Benchmark Metrics

In [ ]:
results = []
detailed_metrics = {}

for name, model in models.items():
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    auc_val = roc_auc_score(y_test, probs) if probs is not None else 0.0
    cm = confusion_matrix(y_test, preds)

    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1,
        'ROC AUC': auc_val
    })
    detailed_metrics[name] = {
        'accuracy': acc, 'precision': prec, 'recall': rec, 'f1_score': f1,
        'roc_auc': auc_val, 'confusion_matrix': cm, 'y_pred': preds, 'y_prob': probs
    }

results_df = pd.DataFrame(results).sort_values(by='Accuracy', ascending=False).reset_index(drop=True)

print("--- ALGORITHM EVALUATION SUMMARY ---")
display(results_df.style.format({
    'Accuracy': '{:.4f}', 'Precision': '{:.4f}', 'Recall': '{:.4f}', 'F1 Score': '{:.4f}', 'ROC AUC': '{:.4f}'
}))


## Step 5: Visualizations (Accuracy Bar Chart, Confusion Matrix, ROC Curves)

In [ ]:
# 1. Bar Chart Comparing Accuracy Across All 5 Algorithms
plt.figure(figsize=(10, 5.5))
colors = ['#2ca02c' if m == 'XGBoost' else '#1f77b4' for m in results_df['Model']]
bars = plt.bar(results_df['Model'], results_df['Accuracy'] * 100, color=colors, width=0.55, edgecolor='black', linewidth=1.2)

plt.title('Algorithm Accuracy Comparison - Loan Eligibility Prediction', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Machine Learning Algorithms', fontsize=12, fontweight='bold')
plt.ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
plt.ylim(60, 100)

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.8, f"{height:.2f}%", ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('model_accuracy_comparison.png', dpi=300)
plt.show()


In [ ]:
# 2. Confusion Matrix & ROC Curve for Best Model (XGBoost)
best_model_name = 'XGBoost'
metrics = detailed_metrics[best_model_name]
cm = metrics['confusion_matrix']
y_prob = metrics['y_prob']

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0],
            xticklabels=['Rejected (0)', 'Approved (1)'], yticklabels=['Rejected (0)', 'Approved (1)'],
            annot_kws={"size": 16, "weight": "bold"})
axes[0].set_title(f'{best_model_name} - Confusion Matrix', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted Label', fontsize=11, fontweight='bold')
axes[0].set_ylabel('True Label', fontsize=11, fontweight='bold')

fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color='#2ca02c', lw=2.5, label=f'{best_model_name} (AUC = {metrics["roc_auc"]:.4f})')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Baseline')
axes[1].set_title(f'{best_model_name} - ROC Curve', fontsize=13, fontweight='bold')
axes[1].set_xlabel('False Positive Rate', fontsize=11, fontweight='bold')
axes[1].set_ylabel('True Positive Rate', fontsize=11, fontweight='bold')
axes[1].legend(loc='lower right', frameon=True, fontsize=11)
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig('xgboost_evaluation.png', dpi=300)
plt.show()


## Step 6: Model Persistence & Inference Demonstration

In [ ]:
# Save trained XGBoost model and preprocessors using joblib
artifacts = {
    'model': models['XGBoost'],
    'scaler': scaler,
    'label_encoders': label_encoders,
    'feature_names': list(X.columns)
}
joblib.dump(artifacts, 'xgboost_loan_model.pkl')
print("[SUCCESS] Best model & preprocessors saved to 'xgboost_loan_model.pkl'.")

# Inference Demonstration
artifacts_loaded = joblib.load('xgboost_loan_model.pkl')
loaded_model = artifacts_loaded['model']
loaded_scaler = artifacts_loaded['scaler']
loaded_les = artifacts_loaded['label_encoders']
feature_names = artifacts_loaded['feature_names']

# New sample applicant profile
sample_applicant = {
    'Gender': 'Male',
    'Married': 'Yes',
    'Dependents': '1',
    'Education': 'Graduate',
    'Self_Employed': 'No',
    'ApplicantIncome': 5000,
    'CoapplicantIncome': 2000,
    'LoanAmount': 150,
    'Loan_Amount_Term': 360,
    'Credit_History': 1.0,
    'Property_Area': 'Urban'
}

sample_df = pd.DataFrame([sample_applicant])

# Transform categorical variables
for col in ['Gender', 'Married', 'Education', 'Self_Employed', 'Property_Area', 'Dependents']:
    sample_df[col] = loaded_les[col].transform(sample_df[col].astype(str))

# Scale numeric variables
sample_df[['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term']] = loaded_scaler.transform(
    sample_df[['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term']]
)

sample_df = sample_df[feature_names]

# Predict
pred = loaded_model.predict(sample_df)[0]
prob = loaded_model.predict_proba(sample_df)[0][1]

status_str = "APPROVED (Y)" if pred == 1 else "REJECTED (N)"
print(f"Sample Applicant Eligibility Prediction: {status_str}")
print(f"Approval Probability: {prob * 100:.2f}%")
